# Simulating strong domain-mix data (cell, spot, and 16µm bin)

This notebook shows how to build the **strong domain-mix** variant of
`albis`'s simulated tissue: a `domain_type_mix` in which every spatial
domain is enriched for two "signature" cell types, rather than the
manuscript-baseline composition used in `xenium_cell_tutorial.ipynb`,
`visium_spot_tutorial.ipynb`, and `visiumHD_bin_tutorial.ipynb`. This is the
dataset behind the manuscript's Figure 3E-G spatial-domain-recovery
benchmark (BANKSY + Harmony + Leiden, evaluated against a sweep of
section-specific batch-effect magnitudes) -- domain identity needs to be
recoverable from cell-type composition alone for that benchmark to be
meaningful, which the baseline composition (several domains only weakly
distinguishable, one exactly uniform) does not reliably provide.

Only `domain_type_mix` changes what gets simulated here -- everything else
(cell placement, gene expression, count noise, batch effects, slicing) works
exactly as in the other three tutorials. Three sections below, one per
modality, each starting from that modality's exact Figure 2 configuration
(same tissue/domain/dispersion tuning) with `domain_type_mix` swapped to the
strong-mix matrix and `batch_sigma` set to that modality's own canonical
value (Cell: 1.5, Spot: 0.3, Bin 16µm: 0.7 -- also the point Figure 3G's
per-modality dotted line marks). To reproduce the *full* batch-effect sweep
behind Figure 3G (not just this one canonical point per modality), see
`sim_paper/code/clustering/strong_mix/generate_strong_mix_{cell,bin16um,spot}.sh`
and `sim_paper/data/figure_3/supplementary_table_sim_parameters_strongmix.tex`.


## Setup

Install `albis` with the notebook + plotting extras, either from a clone of
this repo or from PyPI once published (same two options as the package
`README.md`):

```bash
pip install -e ".[tutorial,plot]"      # from a clone
# or: pip install "albis[tutorial,plot]"
```

Then launch Jupyter:

```bash
jupyter-notebook --no-browser --ip=0.0.0.0 --port 8888
```


In [ ]:
from pathlib import Path
import sys

def find_repo_root(start):
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").is_file() and (candidate / "albis" / "__init__.py").is_file():
            return candidate
        nested = candidate / "albis"
        if (nested / "pyproject.toml").is_file() and (nested / "albis" / "__init__.py").is_file():
            return nested
    raise RuntimeError("Could not find the albis repository root.")

repo_root = find_repo_root(Path.cwd().resolve())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

sys.modules.pop("albis", None)
import albis as ab
import numpy as np

print("Python:", sys.executable)
print("albis module:", getattr(ab, "__file__", "<no __file__>"))
print("albis version:", getattr(ab, "__version__", "<no __version__>"))


## The strong domain-mix composition

Shared by all three sections below.

In [ ]:
# Strong domain-mix cell-type composition (6 domains x 8 cell types). Every
# domain gets exactly two "signature" cell types enriched to 0.30 each
# (2.4x a uniform 0.125) and the remaining 6 types compressed to 0.0667 each
# (~0.53x uniform) -- no domain is left uniform. Pairwise L1 distance between
# domain compositions is 0.467-0.933, vs. just 0.06-0.36 for the
# manuscript-baseline domain_type_mix used in the other three tutorials in
# this folder -- more than 3x the old matrix's *maximum* at its new
# *minimum*. This makes spatial-domain identity recoverable from cell-type
# composition alone, which is what Figure 3's BANKSY domain-recovery
# benchmark (Figure 3E-G) needs: cell type is still drawn i.i.d. per cell
# given its domain (nothing about placement, expression, or noise changes),
# only the per-domain type probabilities become more skewed.
strong_domain_type_mix = np.array([
    [0.30, 0.30, 0.0667, 0.0667, 0.0667, 0.0667, 0.0667, 0.0667],
    [0.0667, 0.0667, 0.30, 0.30, 0.0667, 0.0667, 0.0667, 0.0667],
    [0.0667, 0.0667, 0.0667, 0.0667, 0.30, 0.30, 0.0667, 0.0667],
    [0.0667, 0.0667, 0.0667, 0.0667, 0.0667, 0.0667, 0.30, 0.30],
    [0.30, 0.0667, 0.0667, 0.0667, 0.30, 0.0667, 0.0667, 0.0667],
    [0.0667, 0.0667, 0.30, 0.0667, 0.0667, 0.0667, 0.0667, 0.30],
])


## Shared helper functions

Also shared by all three sections below.

In [ ]:
def fit_empirical_theta(X):
    """Same method count_distribution.py uses to fit an empirical NB dispersion
    from a count matrix, for comparing against real platform data."""
    if hasattr(X, "multiply"):
        mean = np.asarray(X.mean(axis=0)).ravel()
        mean_sq = np.asarray(X.multiply(X).mean(axis=0)).ravel()
    else:
        mean = X.mean(axis=0)
        mean_sq = (X ** 2).mean(axis=0)
    var = mean_sq - mean ** 2
    overdispersed = var > mean
    theta_hat = mean[overdispersed] ** 2 / (var[overdispersed] - mean[overdispersed])
    theta_hat = theta_hat[np.isfinite(theta_hat) & (theta_hat > 0)]
    return float(np.median(theta_hat)) if theta_hat.size else float("nan")


def empirical_domain_composition(adata, n_cell_types=8):
    """Realized per-domain cell-type fractions, straight from obs -- compare
    against strong_domain_type_mix above to confirm the enrichment actually
    landed the way it was specified (domain assignment and cell-type
    assignment happen in two separate steps, so this is worth checking
    directly rather than assuming)."""
    import pandas as pd
    types = [f"type{i}" for i in range(1, n_cell_types + 1)]
    ct = pd.Categorical(adata.obs["cell_type_true"], categories=types)
    table = pd.crosstab(adata.obs["domain_true"], ct)
    return table.div(table.sum(axis=1), axis=0)


## Part 1: Cell (Xenium-like, single-cell resolution)

Starts from `xenium_cell_tutorial.ipynb`'s exact Figure 2 "cell"
configuration, with three changes: `domain_type_mix` -> the strong-mix
matrix above; `sphere_R_um` 6000 -> 2050 and an explicit `n_cells=24_207`
(matching bin16um/spot's disc and Figure 2's own smaller_sphere cell tag --
cell's own native 6000um/600,000-cell scale gives too low a 3D packing
fraction for the domain-recovery benchmark's BANKSY spatial graph); and
`base_gene_lognormal` retuned slightly (-2.3 -> -2.5) alongside that same
2026-09-17 change. `sync_unaligned_seed=True` is also set here (unlike the
baseline cell tutorial) so cell's per-slice misalignment is directly
comparable to bin16um/spot's in the domain-recovery benchmark. Everything
else -- tissue geometry, domain shape, `theta`/`theta_jitter`, marker
fold-changes -- is unchanged from the baseline cell tutorial.

In [ ]:
sphere_r_um = 2050.0          # TUNED (strong-mix only): matches bin16um/spot's disc, not cell's native 6000

config_cell = {
    # ==== 3D tissue geometry and cell placement ====
    "sphere_R_um": sphere_r_um,
    "center": (0.0, 0.0, 0.0),                 # default
    "n_cells": 24_207,                         # TUNED (strong-mix only): matches Figure 2's smaller_sphere cell tag
    "allow_cell_overlap": False,               # default
    "cell_radius_kwargs": dict(                # r_mean/r_sigma tuned; r_min/r_max scaled proportionally
        radius_dist="lognormal", r_mean=7.5, r_sigma=0.28, r_min=4.0, r_max=14.0,
    ),

    # ==== Spatial domains with irregular boundaries (manuscript-wide, all pairings) ====
    "n_domains": 6,
    "core_frac": 0.55,
    "core_bump_amp": 0.25,
    "wedge_angle_amp_deg": 25.0,
    "noise_terms": 16,
    "noise_freq_range": (3.0, 6.0),
    "boundary_fuzz_width_deg": 6.0,
    "boundary_fuzz_flip_prob": 0.15,
    "core_fuzz_width_um": 0.05 * sphere_r_um,  # manuscript ratio: scales with sphere_R_um
    "core_fuzz_flip_prob": 0.25,

    # ==== Cell type composition by domain ====
    "n_cell_types": 8,
    "domain_type_mix": strong_domain_type_mix,  # the whole point of this notebook

    # ==== Gene programs and per-cell gene expression ====
    "marker_genes_per_type": 80,               # default
    "noise_gene_frac": 0.10,                   # default
    "shared_marker_frac": 0.25,                # default
    "base_gene_lognormal": (-2.5, 0.7),        # TUNED: lowers avg. per-cell total counts vs. default (0.7, 0.7)
    "marker_foldchange": 3.5,                  # default
    "shared_marker_foldchange": 2.5,           # default
    "noise_scale": 1.3,                        # TUNED: manuscript "noisy" baseline (default 0.9)
    "cell_size_lognormal": (0.0, 0.35),        # default
    "domain_size_factors": None,               # default (no per-domain expression shift)
    "theta": 0.40,                             # TUNED: NB dispersion (real Xenium fits theta_hat ~0.15-0.16)
    "theta_jitter": 0.15,                      # TUNED: per-gene spread of theta around 0.40

    # ==== Transcript-level molecule simulation ====
    "inside_prob": 0.95,                       # default
    "assign_k": 8,                             # default

    # ==== Platform-specific capture windows ====
    "xenium_capture_window_um": (12000.0, 24000.0),  # real Xenium slide window (12x24mm) -- crops cell output
    "capture_window_um": (6500.0, 6500.0),           # unused for cell output, kept for completeness
    "capture_window_center_um": (0.0, 0.0),          # default
    "bin_size_um": 8.0,                              # unused for cell output
    "spot_spacing_um": 100.0,                        # unused for cell output
    "spot_radius_um": 27.5,                          # unused for cell output

    # ==== Slicing, batch effects, and unaligned coordinates ====
    "n_slices": 10,
    "slice_axes": ("Z",),
    "batch_sigma": 1.5,                        # cell's own canonical/modality-specific batch effect (Figure 3G)
    "max_deg": 270.0,
    "max_shift": 0.5 * sphere_r_um,            # manuscript ratio: scales with sphere_R_um
    "base_seed_unaligned": 12345,              # default
    "sync_unaligned_seed": True,               # TUNED (strong-mix only): comparable misalignment across modalities

    # ==== Simulation outputs ====
    "output_modalities": ("cell",),            # single-cell resolution -- the defining choice for "Xenium-like"
    "sparse_X": True,                          # default
    "seed": 2025,                              # default
}


> **Runtime warning.** This section simulates **24,207 cells** at full
> 3D tissue scale (same scale `sim_paper`'s own strong-mix generation scripts
> use), not a reduced interactive-scale example like
> `higher_level_api_tutorial.ipynb`/`lower_level_api_tutorial.ipynb`.
> If you just want to see the API work end-to-end quickly, drop `n_cells`
> and `sphere_R_um` down (e.g. `n_cells=5000`, proportionally smaller
> `sphere_R_um`) -- every other setting below still applies.


## Generate

In [ ]:
sim_cell = ab.simulate_3d_molecule_sphere_multires(**config_cell)

print("top-level keys:", list(sim_cell.keys()))
adata_cell = sim_cell["adata_cell_sectioned"]["Z"]
adata_cell


## Inspect the output

In [ ]:
print(adata_cell)
print()
print("n_obs (observations):", adata_cell.n_obs)
print("n_vars (genes):", adata_cell.n_vars)
print("obs columns:", list(adata_cell.obs.columns))
print("var columns:", list(adata_cell.var.columns))


In [ ]:
summary = ab.describe(adata_cell)
summary


### Empirical count dispersion and realized domain composition

Two sanity checks worth running whenever you change `domain_type_mix`: the
empirical per-gene NB dispersion (same check `count_distribution.py` runs
against real platform data) is unaffected by domain-mix strength -- it's a
pure expression-noise property -- but the realized per-domain composition
should now closely match `strong_domain_type_mix` above, not the old
manuscript-baseline matrix.

In [ ]:
raw_counts = adata_cell.layers["counts_pre_batch"] if "counts_pre_batch" in adata_cell.layers else adata_cell.X
empirical_theta = fit_empirical_theta(raw_counts)
print(f"empirical theta_hat = {empirical_theta:.3f}")


In [ ]:
realized_composition = empirical_domain_composition(adata_cell)
realized_composition.round(3)


## Plot

`ab.plot()` returns a Matplotlib figure. `coordinates="aligned"` shows the
canonical tissue coordinates (what `domain_true`/`cell_type_true` were
assigned against); `coordinates="unaligned"` shows the per-slice rigid
misalignment `max_deg`/`max_shift` introduce -- the same synthetic
registration-error ground truth used elsewhere in the manuscript. Compare
the `domain_true` panel here against the same panel in the sibling
manuscript-baseline tutorials: composition is now visibly more distinct
domain-to-domain even before any clustering is run.

In [ ]:
fig = ab.plot(adata_cell, view="2d", coordinates="aligned", color="domain_true", point_size=3)
fig


In [ ]:
fig = ab.plot(adata_cell, view="2d", coordinates="aligned", color="cell_type_true", point_size=3)
fig


In [ ]:
fig = ab.plot(adata_cell, view="2d", coordinates="unaligned", color="slice_id", point_size=3)
fig


## Save

Same `ab.save(...)` helper as the other tutorials -- writes an `AnnData` to
`.h5ad` and returns the path.

In [ ]:
output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

output_path_cell = ab.save(adata_cell, output_dir / "strong_mix_cell_z.h5ad")
output_path_cell


## Part 2: Spot (Visium-like)

Starts from `visium_spot_tutorial.ipynb`'s exact Figure 2 "spot"
configuration -- the **only** change here is `domain_type_mix`. Every other
value (tissue geometry, `theta`/`theta_jitter`, `batch_sigma`) is identical
to the baseline spot tutorial; spot's dispersion/geometry tuning never
needed a strong-mix-specific adjustment the way cell's did.

In [ ]:
sphere_r_um = 2050.0          # TUNED: raises 3D packing fraction so spots aren't mostly empty (same as baseline)

config_spot = {
    # ==== 3D tissue geometry and cell placement ====
    "sphere_R_um": sphere_r_um,
    "center": (0.0, 0.0, 0.0),                 # default
    "n_cells": 600_000,                        # manuscript scale
    "allow_cell_overlap": False,               # default
    "cell_radius_kwargs": dict(                # manuscript default (not overridden for spot)
        radius_dist="lognormal", r_mean=7.5, r_sigma=0.28, r_min=4.0, r_max=14.0,
    ),

    # ==== Spatial domains with irregular boundaries (manuscript-wide, all pairings) ====
    "n_domains": 6,
    "core_frac": 0.55,
    "core_bump_amp": 0.25,
    "wedge_angle_amp_deg": 25.0,
    "noise_terms": 16,
    "noise_freq_range": (3.0, 6.0),
    "boundary_fuzz_width_deg": 6.0,
    "boundary_fuzz_flip_prob": 0.15,
    "core_fuzz_width_um": 0.05 * sphere_r_um,  # manuscript ratio: scales with sphere_R_um
    "core_fuzz_flip_prob": 0.25,

    # ==== Cell type composition by domain ====
    "n_cell_types": 8,
    "domain_type_mix": strong_domain_type_mix,  # the whole point of this notebook

    # ==== Gene programs and per-cell gene expression ====
    "marker_genes_per_type": 80,               # default
    "noise_gene_frac": 0.10,                   # default
    "shared_marker_frac": 0.25,                # default
    "base_gene_lognormal": (-2.0, 0.7),        # TUNED: lowers avg. per-cell total counts vs. default (0.7, 0.7)
    "marker_foldchange": 3.5,                  # default
    "shared_marker_foldchange": 2.5,           # default
    "noise_scale": 1.3,                        # TUNED: manuscript "noisy" baseline (default 0.9)
    "cell_size_lognormal": (0.0, 0.35),        # default
    "domain_size_factors": None,               # default (no per-domain expression shift)
    "theta": 0.25,                             # TUNED: NB dispersion, picked against 2 real CytAssist probe refs
    "theta_jitter": 0.10,                      # TUNED: per-gene spread of theta -- see baseline tutorial's note

    # ==== Transcript-level molecule simulation ====
    "inside_prob": 0.95,                       # default
    "assign_k": 8,                             # default

    # ==== Platform-specific capture windows ====
    "xenium_capture_window_um": (12000.0, 24000.0),  # unused for spot output
    "capture_window_um": (6500.0, 6500.0),           # real Visium slide window (6.5x6.5mm) -- crops spot output
    "capture_window_center_um": (0.0, 0.0),          # default
    "bin_size_um": 8.0,                              # unused for spot output
    "spot_spacing_um": 100.0,                        # default -- Visium spot center-to-center spacing
    "spot_radius_um": 27.5,                          # default -- Visium spot capture radius

    # ==== Slicing, batch effects, and unaligned coordinates ====
    "n_slices": 10,
    "slice_axes": ("Z",),
    "batch_sigma": 0.3,                        # spot's own canonical/modality-specific batch effect (Figure 3G)
    "max_deg": 270.0,
    "max_shift": 0.5 * sphere_r_um,            # manuscript ratio: scales with sphere_R_um
    "base_seed_unaligned": 12345,              # default

    # ==== Simulation outputs ====
    "output_modalities": ("spot",),            # spot-resolution aggregation -- the defining choice for "Visium-like"
    "sparse_X": True,                          # default
    "seed": 2025,                              # default
}


> **Runtime warning.** This section simulates **600,000 cells** at full
> 3D tissue scale (same scale `sim_paper`'s own strong-mix generation scripts
> use), not a reduced interactive-scale example like
> `higher_level_api_tutorial.ipynb`/`lower_level_api_tutorial.ipynb`. 600,000 cells are still
> simulated at full 3D tissue scale before being aggregated down to spots,
> so this is not meaningfully faster to *generate* than the cell-resolution
> section above, even though the final spot-level output has far fewer
> observations.
> If you just want to see the API work end-to-end quickly, drop `n_cells`
> and `sphere_R_um` down (e.g. `n_cells=5000`, proportionally smaller
> `sphere_R_um`) -- every other setting below still applies.


## Generate

In [ ]:
sim_spot = ab.simulate_3d_molecule_sphere_multires(**config_spot)

print("top-level keys:", list(sim_spot.keys()))
adata_spot = sim_spot["spot_adatas"]["Z"]
adata_spot


## Inspect the output

In [ ]:
print(adata_spot)
print()
print("n_obs (observations):", adata_spot.n_obs)
print("n_vars (genes):", adata_spot.n_vars)
print("obs columns:", list(adata_spot.obs.columns))
print("var columns:", list(adata_spot.var.columns))


In [ ]:
summary = ab.describe(adata_spot)
summary


### Empirical count dispersion and realized domain composition

Two sanity checks worth running whenever you change `domain_type_mix`: the
empirical per-gene NB dispersion (same check `count_distribution.py` runs
against real platform data) is unaffected by domain-mix strength -- it's a
pure expression-noise property -- but the realized per-domain composition
should now closely match `strong_domain_type_mix` above, not the old
manuscript-baseline matrix.

In [ ]:
raw_counts = adata_spot.layers["counts_pre_batch"] if "counts_pre_batch" in adata_spot.layers else adata_spot.X
empirical_theta = fit_empirical_theta(raw_counts)
print(f"empirical theta_hat = {empirical_theta:.3f}")


In [ ]:
realized_composition = empirical_domain_composition(adata_spot)
realized_composition.round(3)


## Plot

`ab.plot()` returns a Matplotlib figure. `coordinates="aligned"` shows the
canonical tissue coordinates (what `domain_true`/`cell_type_true` were
assigned against); `coordinates="unaligned"` shows the per-slice rigid
misalignment `max_deg`/`max_shift` introduce -- the same synthetic
registration-error ground truth used elsewhere in the manuscript. Compare
the `domain_true` panel here against the same panel in the sibling
manuscript-baseline tutorials: composition is now visibly more distinct
domain-to-domain even before any clustering is run.

In [ ]:
fig = ab.plot(adata_spot, view="2d", coordinates="aligned", color="domain_true", point_size=3)
fig


In [ ]:
fig = ab.plot(adata_spot, view="2d", coordinates="aligned", color="cell_type_true", point_size=3)
fig


In [ ]:
fig = ab.plot(adata_spot, view="2d", coordinates="unaligned", color="slice_id", point_size=3)
fig


## Save

In [ ]:
output_path_spot = ab.save(adata_spot, output_dir / "strong_mix_spot_z.h5ad")
output_path_spot


## Part 3: Bin, 16µm (Visium HD-like)

Starts from `visiumHD_bin_tutorial.ipynb`'s exact Figure 2 "bin, 16µm"
configuration (built there as the 8µm config with `bin_size_um`,
`base_gene_lognormal`, and `batch_sigma` overridden for the larger bin
size). As with spot, the **only** change from that baseline needed here is
`domain_type_mix`.

In [ ]:
sphere_r_um = 2050.0          # TUNED: raises 3D packing fraction so bins aren't mostly empty (same as baseline)

config_bin16 = {
    # ==== 3D tissue geometry and cell placement ====
    "sphere_R_um": sphere_r_um,
    "center": (0.0, 0.0, 0.0),                 # default
    "n_cells": 600_000,                        # manuscript scale
    "allow_cell_overlap": False,               # default
    "cell_radius_kwargs": dict(                # manuscript default (not overridden for bin)
        radius_dist="lognormal", r_mean=7.5, r_sigma=0.28, r_min=4.0, r_max=14.0,
    ),

    # ==== Spatial domains with irregular boundaries (manuscript-wide, all pairings) ====
    "n_domains": 6,
    "core_frac": 0.55,
    "core_bump_amp": 0.25,
    "wedge_angle_amp_deg": 25.0,
    "noise_terms": 16,
    "noise_freq_range": (3.0, 6.0),
    "boundary_fuzz_width_deg": 6.0,
    "boundary_fuzz_flip_prob": 0.15,
    "core_fuzz_width_um": 0.05 * sphere_r_um,  # manuscript ratio: scales with sphere_R_um
    "core_fuzz_flip_prob": 0.25,

    # ==== Cell type composition by domain ====
    "n_cell_types": 8,
    "domain_type_mix": strong_domain_type_mix,  # the whole point of this notebook

    # ==== Gene programs and per-cell gene expression ====
    "marker_genes_per_type": 80,               # default
    "noise_gene_frac": 0.10,                   # default
    "shared_marker_frac": 0.25,                # default
    "base_gene_lognormal": (-2.5, 0.7),        # TUNED: re-tuned for 16um bins' larger per-bin count scale
    "marker_foldchange": 3.5,                  # default
    "shared_marker_foldchange": 2.5,           # default
    "noise_scale": 1.3,                        # TUNED: manuscript "noisy" baseline (default 0.9)
    "cell_size_lognormal": (0.0, 0.35),        # default
    "domain_size_factors": None,               # default (no per-domain expression shift)
    "theta": 2.0,                              # NOT overridden for bin -- manuscript "noisy" default
    "theta_jitter": 1.0,                       # NOT overridden for bin -- manuscript "noisy" default

    # ==== Transcript-level molecule simulation ====
    "inside_prob": 0.95,                       # default
    "assign_k": 8,                             # default

    # ==== Platform-specific capture windows ====
    "xenium_capture_window_um": (12000.0, 24000.0),  # unused for bin output
    "capture_window_um": (6500.0, 6500.0),           # real Visium HD slide window (6.5x6.5mm) -- crops bin output
    "capture_window_center_um": (0.0, 0.0),          # default
    "bin_size_um": 16.0,                             # Visium HD 16um bin grid -- the defining choice for this section
    "spot_spacing_um": 100.0,                        # unused for bin output
    "spot_radius_um": 27.5,                          # unused for bin output

    # ==== Slicing, batch effects, and unaligned coordinates ====
    "n_slices": 10,
    "slice_axes": ("Z",),
    "batch_sigma": 0.7,                        # bin16um's own canonical/modality-specific batch effect (Figure 3G)
    "max_deg": 270.0,
    "max_shift": 0.5 * sphere_r_um,            # manuscript ratio: scales with sphere_R_um
    "base_seed_unaligned": 12345,              # default

    # ==== Simulation outputs ====
    "output_modalities": ("bin",),             # bin-resolution aggregation -- the defining choice for "VisiumHD-like"
    "sparse_X": True,                          # default
    "seed": 2025,                              # default
}


> **Runtime warning.** This section simulates **600,000 cells** at full
> 3D tissue scale (same scale `sim_paper`'s own strong-mix generation scripts
> use), not a reduced interactive-scale example like
> `higher_level_api_tutorial.ipynb`/`lower_level_api_tutorial.ipynb`. 600,000 cells are
> simulated at full 3D tissue scale before being aggregated into bins, same
> cost as the spot section above regardless of the much smaller final
> bin-level `n_obs`.
> If you just want to see the API work end-to-end quickly, drop `n_cells`
> and `sphere_R_um` down (e.g. `n_cells=5000`, proportionally smaller
> `sphere_R_um`) -- every other setting below still applies.


## Generate

In [ ]:
sim_bin16 = ab.simulate_3d_molecule_sphere_multires(**config_bin16)

print("top-level keys:", list(sim_bin16.keys()))
adata_bin16 = sim_bin16["bin_adatas"]["Z"]
adata_bin16


## Inspect the output

In [ ]:
print(adata_bin16)
print()
print("n_obs (observations):", adata_bin16.n_obs)
print("n_vars (genes):", adata_bin16.n_vars)
print("obs columns:", list(adata_bin16.obs.columns))
print("var columns:", list(adata_bin16.var.columns))


In [ ]:
summary = ab.describe(adata_bin16)
summary


### Empirical count dispersion and realized domain composition

Two sanity checks worth running whenever you change `domain_type_mix`: the
empirical per-gene NB dispersion (same check `count_distribution.py` runs
against real platform data) is unaffected by domain-mix strength -- it's a
pure expression-noise property -- but the realized per-domain composition
should now closely match `strong_domain_type_mix` above, not the old
manuscript-baseline matrix.

In [ ]:
raw_counts = adata_bin16.layers["counts_pre_batch"] if "counts_pre_batch" in adata_bin16.layers else adata_bin16.X
empirical_theta = fit_empirical_theta(raw_counts)
print(f"empirical theta_hat = {empirical_theta:.3f}")


In [ ]:
realized_composition = empirical_domain_composition(adata_bin16)
realized_composition.round(3)


## Plot

`ab.plot()` returns a Matplotlib figure. `coordinates="aligned"` shows the
canonical tissue coordinates (what `domain_true`/`cell_type_true` were
assigned against); `coordinates="unaligned"` shows the per-slice rigid
misalignment `max_deg`/`max_shift` introduce -- the same synthetic
registration-error ground truth used elsewhere in the manuscript. Compare
the `domain_true` panel here against the same panel in the sibling
manuscript-baseline tutorials: composition is now visibly more distinct
domain-to-domain even before any clustering is run.

In [ ]:
fig = ab.plot(adata_bin16, view="2d", coordinates="aligned", color="domain_true", point_size=3)
fig


In [ ]:
fig = ab.plot(adata_bin16, view="2d", coordinates="aligned", color="cell_type_true", point_size=3)
fig


In [ ]:
fig = ab.plot(adata_bin16, view="2d", coordinates="unaligned", color="slice_id", point_size=3)
fig


## Save

In [ ]:
output_path_bin16 = ab.save(adata_bin16, output_dir / "strong_mix_bin16um_z.h5ad")
output_path_bin16


## Next steps

These three `.h5ad` files are single-canonical-batch_sigma points -- exactly
one point each on Figure 3G's per-modality curve. To reproduce the *full*
batch-effect sweep (the actual Figure 3E-G analysis: BANKSY -> PCA -> Harmony
-> Leiden, scored against `domain_true` by adjusted Rand index across many
`batch_sigma` values per modality), see:

- `sim_paper/code/clustering/strong_mix/generate_strong_mix_cell.sh` (9-point sweep)
- `sim_paper/code/clustering/strong_mix/generate_strong_mix_bin16um.sh` (7-point sweep)
- `sim_paper/code/clustering/strong_mix/generate_strong_mix_spot.sh` (16-point sweep)
- `sim_paper/data/figure_3/supplementary_table_sim_parameters_strongmix.tex` for the full parameter table
